# 05 · Batch inference with Spark

Runs the fine-tuned AraBERT model over the whole test set locally on CPU, distributing the work across Spark workers with `mapInPandas`.

| | |
|---|---|
| **Input** | Test split from `data/silver/hard_reviews/`, model from the Hugging Face Hub |
| **Output** | `data/gold/arabert_test_predictions` (`review_id`, `ar_prob_pos`, `ar_pred`) |

## Log in to Hugging Face

The model repo is private, so a token is needed. It is entered with `getpass` and never stored in the notebook.

In [9]:
from getpass import getpass
from huggingface_hub import login, logout

try:
    logout()
except Exception:
    pass

login(token=getpass("HF token: "))

## Download the model and load the test set

`snapshot_download` saves the model to the local Hugging Face cache once, so every Spark worker can load it from disk instead of downloading it again.

In [10]:
import time
from config import get_spark, SILVER_PATH, GOLD_PATH
from pyspark.sql import functions as F
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download("Mahmoud-103/arabert-hard-sentiment")
print(MODEL_DIR)

spark = get_spark("inference")

test = (spark.read.parquet(SILVER_PATH)
        .filter("split = 'test'")
        .select(F.col("review_id").cast("int"), "review_clean", "label"))
print("test rows:", test.count())

test rows: 10217


## Prediction function

`mapInPandas` hands each worker its partition as a stream of pandas DataFrames. The function:

1. Loads the tokenizer and model **once per partition**, not once per row.
2. Scores reviews in batches of 32, using the same truncation (128 tokens) as training.
3. Returns the probability of the positive class and the predicted label (threshold 0.5).

`torch.set_num_threads(2)` stops the parallel workers from fighting over the same CPU cores.

In [11]:
def predict_partition(batches):
 
    import torch
    import pandas as pd
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    torch.set_num_threads(2)

    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()

    for pdf in batches:
        texts = pdf["review_clean"].tolist()
        probs = []
        for i in range(0, len(texts), 32):
            enc = tok(texts[i:i + 32], truncation=True, max_length=128,
                      padding=True, return_tensors="pt")
            with torch.no_grad():
                p = torch.softmax(model(**enc).logits, dim=-1)[:, 1]
            probs.extend(p.tolist())

        yield pd.DataFrame({
            "review_id":   pdf["review_id"].values,
            "ar_prob_pos": probs,
            "ar_pred":     [int(x >= 0.5) for x in probs],
        })

OUT_SCHEMA = "review_id int, ar_prob_pos double, ar_pred int"

## Smoke test

Run 200 rows first to catch errors and estimate the speed before launching the full job.

In [12]:
t0 = time.time()
sample = test.limit(200).repartition(2).mapInPandas(predict_partition, schema=OUT_SCHEMA)
sample.show(5)
print(f"200 rows took {time.time() - t0:.0f}s")

+---------+-------------------+-------+
|review_id|        ar_prob_pos|ar_pred|
+---------+-------------------+-------+
|   204816| 0.8597769141197205|      1|
|   225863|0.04026075825095177|      0|
|    41668|  0.978097140789032|      1|
|    92677| 0.9825819730758667|      1|
|   373338|0.22962214052677155|      0|
+---------+-------------------+-------+
only showing top 5 rows
200 rows took 39s


## Full run

The test set is split into one partition per worker: half the CPU cores (8 on this machine), each worker using 2 threads, so all 16 cores are busy. 10,217 reviews took about 10 minutes (17.6 reviews/sec).

In [13]:
import os
print("CPU cores:", os.cpu_count())

CPU cores: 16


In [14]:
N_WORKERS = max(2, os.cpu_count() // 2)
PRED_PATH = f"{GOLD_PATH}/arabert_test_predictions"

t0 = time.time()
(test.repartition(N_WORKERS)
     .mapInPandas(predict_partition, schema=OUT_SCHEMA)
     .write.mode("overwrite").parquet(PRED_PATH))
elapsed = time.time() - t0
print(f"{N_WORKERS} workers | {elapsed/60:.1f} min | {10217/elapsed:.1f} reviews/sec")

8 workers | 9.7 min | 17.6 reviews/sec


## Test accuracy

Join the predictions back to the true labels: **91.85% test accuracy**, matching val (91.91%). All 10,217 rows came back, so no partition was lost.

In [15]:
preds = spark.read.parquet(PRED_PATH)
joined = test.join(preds, on="review_id")

acc = joined.select(F.avg((F.col("label") == F.col("ar_pred")).cast("int"))).first()[0]
print(f"AraBERT TEST accuracy: {acc:.4f}   (rows: {joined.count()})")

AraBERT TEST accuracy: 0.9185   (rows: 10217)
